# Advanced OOP Topics


In [ ]:
import sys
from pathlib import Path

current = Path.cwd()
for parent in [current, *current.parents]:
    if (parent / '_config.yml').exists():
        project_root = parent
        break
else:
    project_root = Path.cwd().parent.parent

sys.path.insert(0, str(project_root))

from shared import thinkpython, diagram, jupyturtle, download

sys.modules['thinkpython'] = thinkpython
sys.modules['diagram'] = diagram
sys.modules['jupyturtle'] = jupyturtle
sys.modules['download'] = download


This notebook covers four topics that extend the core OOP material:

| Topic | What it adds |
|---|---|
| Comparison dunder methods | Sorting, sets, and dictionary keys for custom objects |
| `@dataclass` in depth | Auto-generated `__init__`, `__repr__`, `__eq__`, ordering, frozen instances |
| Abstract Base Classes | Formalizing interfaces with `ABC` and `@abstractmethod` |
| Class vs. instance variables | A common source of bugs, explained clearly |


## Comparison Methods: `__eq__`, `__lt__`, `__hash__`


By default, `==` on a custom object tests **identity** (same as `is`), not value equality.
Defining `__eq__` changes that behavior.


In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x}, {self.y})'

p1 = Point(1, 2)
p2 = Point(1, 2)

print(p1 == p2)   # False — identity check by default
print(p1 is p2)   # False


Adding `__eq__` makes `==` compare by value instead.


In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x}, {self.y})'

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)

print(p1 == p2)   # True
print(p1 == p3)   # False


### Ordering with `__lt__`

To support sorting (`sorted()`, `min()`, `max()`), define `__lt__` (less than).
Python uses it to derive the other comparison operators when you also apply
`@functools.total_ordering`.


In [ ]:
from functools import total_ordering
import math

@total_ordering
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x}, {self.y})'

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __lt__(self, other):
        """Sort by distance from the origin."""
        return math.hypot(self.x, self.y) < math.hypot(other.x, other.y)

points = [Point(3, 4), Point(1, 1), Point(0, 2)]
print(sorted(points))   # sorted by distance from origin
print(min(points))


### Hashability and `__hash__`

When you define `__eq__`, Python **automatically sets `__hash__` to `None`**,
making instances unhashable (cannot be used in sets or as dict keys).
Define `__hash__` explicitly to restore that ability.

```python
# Rule of thumb: objects that compare equal must have the same hash.
def __hash__(self):
    return hash((self.x, self.y))
```

If your object is **mutable**, do not define `__hash__` — mutable objects
should not be hashed because their value (and their hash) could change after insertion.


In [ ]:
from functools import total_ordering
import math

@total_ordering
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x}, {self.y})'

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __lt__(self, other):
        return math.hypot(self.x, self.y) < math.hypot(other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(3, 4)

point_set = {p1, p2, p3}
print(point_set)          # p1 and p2 are equal — only one appears

lookup = {p1: 'origin-ish', p3: 'far'}
print(lookup[p2])         # works because p2 == p1 and hash(p2) == hash(p1)


## `@dataclass` In Depth


You saw `@dataclass` in the previous notebook as a way to eliminate boilerplate.
Here we look at its more useful options.

| Option | Effect |
|---|---|
| `eq=True` (default) | Auto-generates `__eq__` based on fields |
| `order=True` | Also generates `__lt__`, `__le__`, `__gt__`, `__ge__` |
| `frozen=True` | Makes instances immutable; also enables `__hash__` |
| `field(default_factory=...)` | Safe default for mutable fields like lists |


In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Student:
    name: str
    gpa: float
    courses: list = field(default_factory=list)   # safe mutable default

s1 = Student('Alice', 3.8)
s2 = Student('Bob', 3.5)
s3 = Student('Alice', 3.8)

print(s1 == s3)          # True — same field values
print(sorted([s1, s2]))  # sorted lexicographically by (name, gpa)
s1.courses.append('CS101')
print(s1)


### Frozen Dataclasses

`frozen=True` prevents attribute mutation after creation and automatically
provides a correct `__hash__`, making instances usable in sets and as dict keys.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int

red = Color(255, 0, 0)
print(red)
print(hash(red))   # hashable

palette = {red, Color(0, 255, 0), Color(0, 0, 255)}
print(palette)

try:
    red.r = 128    # raises FrozenInstanceError
except Exception as e:
    print(type(e).__name__, e)


## Abstract Base Classes (ABC)


In `1202-oop-practice` you saw inheritance and polymorphism.
An **Abstract Base Class (ABC)** formalizes that pattern by declaring a
*contract*: any subclass **must** implement certain methods, or Python will
refuse to instantiate it.

This makes the interface explicit rather than implied.


In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract base class — cannot be instantiated directly."""

    @abstractmethod
    def area(self) -> float:
        """Return the area of the shape."""

    @abstractmethod
    def perimeter(self) -> float:
        """Return the perimeter of the shape."""

    def describe(self):
        """Concrete method shared by all subclasses."""
        return f'{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}'


try:
    s = Shape()
except TypeError as e:
    print(e)


Concrete subclasses must implement every `@abstractmethod`.


In [ ]:
import math

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


shapes = [Circle(5), Rectangle(4, 6)]
for s in shapes:
    print(s.describe())


Because both classes satisfy the `Shape` contract, code that works on a
`Shape` works on any subclass — this is **polymorphism with a guaranteed interface**.

**When to use ABCs:**
- When you want to enforce that subclasses implement required methods
- When building a library with a plug-in architecture
- As a documentation tool — the abstract methods serve as a spec


## Class Variables vs. Instance Variables


A **class variable** is defined directly in the class body, outside any method.
It is shared across all instances. An **instance variable** is set on `self`
inside a method and belongs only to that one object.

Confusing the two is one of the most common OOP bugs in Python.


In [ ]:
class Dog:
    species = 'Canis lupus familiaris'   # class variable — shared by all dogs

    def __init__(self, name):
        self.name = name                  # instance variable — unique per dog

d1 = Dog('Rex')
d2 = Dog('Fido')

print(d1.species)    # 'Canis lupus familiaris'
print(d2.species)    # same — shared
print(d1.name)       # 'Rex'
print(d2.name)       # 'Fido'


### The Mutation Trap

Assigning to a class variable **via an instance** creates a new instance
variable that *shadows* the class variable — it does **not** change the class
variable for all instances.

But **mutating** a mutable class variable (like a list) *does* affect all
instances, because no new variable is created.


In [ ]:
class Counter:
    count = 0            # class variable
    history = []         # mutable class variable — danger zone

    def __init__(self, name):
        self.name = name

a = Counter('a')
b = Counter('b')

# Reassignment via instance — creates a new instance variable on `a` only
a.count = 99
print(a.count)           # 99  — instance variable on a
print(b.count)           # 0   — class variable unchanged
print(Counter.count)     # 0

# Mutation via instance — modifies the shared class-level list
a.history.append('event')
print(b.history)         # ['event'] — b sees the change!
print(Counter.history)   # ['event']


**Rule of thumb:**

- Use class variables for constants or data that *truly* belongs to the class (e.g., `species`, `MAX_SIZE`).
- Use instance variables (set in `__init__`) for data that belongs to individual objects.
- Never use a mutable class variable as a default container — use `field(default_factory=list)` with `@dataclass`, or set the list in `__init__`.


## Summary

| Topic | Key takeaway |
|---|---|
| `__eq__` / `__lt__` / `__hash__` | Define these to make objects sortable and hashable; remember the mutability rule for `__hash__` |
| `@dataclass` | Use `order=True` for sorting, `frozen=True` for hashable immutable objects, `field(default_factory=...)` for mutable defaults |
| `ABC` | Declare required methods with `@abstractmethod`; prevents incomplete subclasses from being instantiated |
| Class vs. instance variables | Keep mutable state in instance variables; treat class variables as shared constants |
